In [1]:
import os
from pathlib import Path
# Change cwd to the project root (parent of 'notebooks/')
os.chdir(Path.cwd().parent)
Path.cwd()

PosixPath('/Users/jbrandt/code/birddog')

In [2]:
from birddog.database import (
    Database,
)
from birddog.nocodb_database import (
    clone_table_schema,
    copy_records,
    rename_field,
    copy_formula_field,
    list_formula_fields,
    list_lookup_fields,
    create_lookup_field,
)

2026-08-14 18:47:05,424 [INFO] Using local nocodb api: http://localhost:8080


In [ ]:
local_db = Database()
local_db._host

In [3]:
nocodb_aws_host = os.environ["BIRDDOG_AWS_NOCODB_HOST"]
nocodb_aws_token = os.environ["BIRDDOG_AWS_NOCODB_API_TOKEN"]
nocodb_aws_base_id = os.environ["BIRDDOG_AWS_BASE_ID"]

In [4]:
#nocodb_aws_host = nocodb_aws_host.replace("https", "http")
#nocodb_aws_host

In [5]:
aws_db = Database(host=nocodb_aws_host, api_token=nocodb_aws_token, base_id=nocodb_aws_base_id)
aws_db._host

2026-08-14 18:47:18,148 [INFO] creating NocoDBDatabase(host=http://nocodb-env.eba-xhmfyydr.us-east-2.elasticbeanstalk.com, base_id=prsjtz30iuhk88f) instance
2026-08-14 18:47:18,401 [INFO] 
service throttle report:
  host_key                            cfg_rps  act_rps   tokens  blocked_s max_in_flight
  ----------------------------------------------------------------------------------------
  nocodb.internal:api                   20.00     3.99    39.00       0.00           24


'http://nocodb-env.eba-xhmfyydr.us-east-2.elasticbeanstalk.com'

In [ ]:
def _delete_all(db, table_name):
    db.delete(table_name, db.get_all_ids(table_name))
    
def clone_db(src_db, dest_db):
    try:
        clone_table_schema(src_db, "Schema", dest_db, "Schema")
    except ValueError as err:
        print("Schema table already exists")
    try:    
        clone_table_schema(src_db, "Schema Values", dest_db, "Schema Values")
    except ValueError as err:
        print("Schema Values table already exists")

    _delete_all(dest_db, "Schema")
    copy_records(src_db, "Schema", dest_db, "Schema")

    _delete_all(dest_db, "Schema Values")
    copy_records(src_db, "Schema Values", dest_db, "Schema Values")

    dest_db.load_schema()
    clone_table_schema(src_db, "Pages", dest_db, "Pages")
    clone_table_schema(src_db, "Documents", dest_db, "Documents")
    clone_table_schema(src_db, "MasterFileSummary", dest_db, "MasterFileSummary")
    clone_table_schema(src_db, "FondSummary", dest_db, "FondSummary")
    clone_table_schema(src_db, "OpusSummary", dest_db, "OpusSummary")
    dest_db.load_schema()

In [ ]:
clone_db(aws_db, local_db)

In [ ]:
rename_field(local_db, local_db._field_id("Pages", "Pages"), "parent")
rename_field(local_db, local_db._field_id("Documents", "Pages"), "owning_pages")
local_db.load_schema()

In [ ]:
list_lookup_fields(aws_db, "Documents")

In [ ]:
def list_rollup_fields(db, table):
    info = db._get_table_info(table)
    return [c["title"] for c in info["columns"] if c["uidt"] == "Rollup"]

In [ ]:
list_rollup_fields(aws_db, "OpusSummary")

In [ ]:
rename_field(local_db, local_db._field_id("Documents", "Pages"), "owning_pages")

In [6]:
rename_field(aws_db, "c9bgrdnnxmr90et", "duplicate_of")

2026-08-14 18:48:25,820 [INFO] 
service throttle report:
  host_key                            cfg_rps  act_rps   tokens  blocked_s max_in_flight
  ----------------------------------------------------------------------------------------
  nocodb.internal:api                   21.00     0.04    39.00       0.00           24


In [ ]:
local_db.load_schema()


In [ ]:
list_formula_fields(aws_db, "Pages")

In [ ]:
copy_formula_field(aws_db, "Pages", "root_label", local_db, "Pages", "root_label")

In [ ]:
list_formula_fields(db, "Documents OLD")

In [ ]:
r=copy_formula(db, "Documents OLD", "domain", db, "Documents", "domain")

In [ ]:
list_lookup_fields(db,"Documents OLD")

In [ ]:
ci = db._get_table_info("Documents OLD")["columns"]

In [ ]:
for c in ci:
    if c["uidt"] == "Lookup":
        print(c["title"])
        print(c)
        break

In [ ]:
db._get_table_id_map()

In [ ]:
l=list_lookup_fields(db,"Documents OLD")

In [ ]:
#r=create_lookup_field(db, "Documents", "root", "owning_pages", "Pages", "root")

In [ ]:
for f in l[1:]:
    create_lookup_field(db, "Documents", f, "owning_pages", "Pages", f)

In [ ]:
r=create_lookup_field(db, "Documents", "page_description", "owning_pages", "Pages", "description")

In [ ]:
r=create_lookup_field(db, "Documents", "native_page_description", "owning_pages", "Pages", "native_description")

In [ ]:
l

In [ ]:
list_lookup_fields(db,"Documents OLD")

In [ ]:
list_lookup_fields(db,"Documents")

In [ ]:
list_lookup_fields(db,"Pages")

In [ ]:
list_formula_fields(db, "Documents OLD")